# Initial Setup


In [1]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

Looking in links: /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arc_agi-0.9.8-py3-none-any.whl
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/arcengine-0.9.3-py3-none-any.whl (from arc-agi)
Processing /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels/pillow-12.2.0-cp312-cp312-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (from arc-agi)
  Attempting uninstall: pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
gradio 5.50.0 requires pillow<12.0,>=8.0, but you have pillow 12.2.0 which is incompatib

In [3]:
import numpy as np 
import pandas as pd
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub
print("All Packages Loaded successfully!")

/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/sk48/d8078629/sk48.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/sk48/d8078629/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/tn36/ef4dde99/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/tn36/ef4dde99/tn36.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/m0r0/492f87ba/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/m0r0/492f87ba/m0r0.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/bp35/0a0ad940/bp35.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/bp35/0a0ad940/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/cn04/2fe56bfb/cn04.py
/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files/cn04/2fe56bfb/metadata.json
/kaggle/input/competitions/arc-prize-2026-arc-agi-

# Importing Necessary Wheels

In [ ]:
!pip install --no-index --find-links=/kaggle/input/notebooks/banwait13/datasets-for-arc-agi/wheels sentence-transformers faiss-cpu 

# Writing the Core Logic for Agent

In [ ]:
%%writefile /kaggle/working/my_agent.py
import os
import subprocess
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch as t
from torch.nn import nn

import random
import time
from typing import Any, List, Tuple, Dict
from dataclasses import dataclass

from arcengine import FrameData, GameAction, GameState
from agents.agent import Agent


class Grid(*args, **kwargs) -> None{

    
}


class MyAgent(Agent):
    """
    The Empty Shell. 
    It just picks a random action so the game doesn't crash.
    """
    MAX_ACTIONS = float('inf')

    def __init__(self, *args: Any, **kwargs: Any) -> None:
        super().__init__(*args, **kwargs)
        seed = int(time.time() * 1000000) + hash(self.game_id) % 1000000
        random.seed(seed)

    def is_done(self, frames: list[FrameData], latest_frame: FrameData) -> bool:
        # Stop playing if we win!
        return latest_frame.state is GameState.WIN

    def choose_action(self, frames: list[FrameData], latest_frame: FrameData) -> GameAction:
        
        # ==========================================
        # YOUR FUTURE BRAIN LOGIC WILL GO HERE!
        # ==========================================
        
        # If the game is over or hasn't started, press reset.
        if latest_frame.state in [GameState.NOT_PLAYED, GameState.GAME_OVER]:
            action = GameAction.RESET
        else:
            # Otherwise, just smash a random button.
            action = random.choice([a for a in GameAction if a is not GameAction.RESET])

        return action

# Submission

In [ ]:
# 1. IF KAGGLE IS RUNNING THE OFFICIAL TEST:
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    
    # Wait for the hidden game server to wake up
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    # Copy the agent framework into our working folder
    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    # Inject our custom agent (from Cell 2) into their framework
    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    # Override the default settings to point to our agent
    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("""from typing import Type, cast
from dotenv import load_dotenv
from .agent import Agent, Playback
from .swarm import Swarm
from .templates.random_agent import Random
from .templates.my_agent import MyAgent

load_dotenv()

AVAILABLE_AGENTS: dict[str, Type[Agent]] = {
    "random": Random,
    "myagent": MyAgent,
}
""")

    # Setup the offline network connections
    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("""SCHEME=http
HOST=gateway
PORT=8001
ARC_API_KEY=test-key-123
ARC_BASE_URL=http://gateway:8001/
OPERATION_MODE=online
ENVIRONMENTS_DIR=
RECORDINGS_DIR=/kaggle/working/server_recording
""")

    # Fire up the agent!
    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent

# 2. IF YOU ARE JUST SAVING THE NOTEBOOK WHILE CODING:
else:
    # Make a dummy submission file so Kaggle doesn't crash on save
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
    print("Created dummy submission file. Setup complete!")